In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

flight = pd.read_csv('flights.csv')

## Data preview and basic information
Each row represents a flight. The previews below retain the original data; negative delays indicate early departures or arrivals.

In [2]:
from IPython.display import display

# First 10 rows, with every column visible
with pd.option_context("display.max_columns", None):
    display(flight.head(10))

,id,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,name
0,0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01 05:00:00,United Air Lines Inc.
1,1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01 05:00:00,United Air Lines Inc.
2,2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01 05:00:00,American Airlines Inc.
3,3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01 05:00:00,JetBlue Airways
4,4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01 06:00:00,Delta Air Lines Inc.
5,5,2013,1,1,554.0,558,-4.0,740.0,728,12.0,UA,1696,N39463,EWR,ORD,150.0,719,5,58,2013-01-01 05:00:00,United Air Lines Inc.
6,6,2013,1,1,555.0,600,-5.0,913.0,854,19.0,B6,507,N516JB,EWR,FLL,158.0,1065,6,0,2013-01-01 06:00:00,JetBlue Airways
7,7,2013,1,1,557.0,600,-3.0,709.0,723,-14.0,EV,5708,N829AS,LGA,IAD,53.0,229,6,0,2013-01-01 06:00:00,ExpressJet Airlines Inc.
8,8,2013,1,1,557.0,600,-3.0,838.0,846,-8.0,B6,79,N593JB,JFK,MCO,140.0,944,6,0,2013-01-01 06:00:00,JetBlue Airways
9,9,2013,1,1,558.0,600,-2.0,753.0,745,8.0,AA,301,N3ALAA,LGA,ORD,138.0,733,6,0,2013-01-01 06:00:00,American Airlines Inc.


In [3]:
# A reproducible sample from across the dataset
preview_columns = ["year", "month", "day", "carrier", "flight", "origin", "dest", "dep_delay", "arr_delay", "air_time", "distance"]
display(flight.sample(10, random_state=42)[preview_columns])

,year,month,day,carrier,flight,origin,dest,dep_delay,arr_delay,air_time,distance
159280,2013,3,25,UA,1471,EWR,RSW,24.0,19.0,169.0,1068
189101,2013,4,26,DL,1765,JFK,SFO,-4.0,-37.0,337.0,2586
212435,2013,5,21,EV,4129,EWR,DCA,11.0,16.0,39.0,199
266804,2013,7,18,EV,5796,EWR,CLT,-8.0,-22.0,77.0,529
306581,2013,8,29,B6,939,JFK,BQN,-5.0,0.0,198.0,1576
68828,2013,11,14,UA,314,JFK,LAX,6.0,2.0,342.0,2475
164026,2013,3,30,B6,675,JFK,LAX,0.0,-4.0,338.0,2475
289110,2013,8,10,DL,4,JFK,SJU,-1.0,2.0,205.0,1598
161286,2013,3,27,9E,4091,JFK,BWI,-5.0,-9.0,37.0,184
205307,2013,5,13,DL,245,JFK,PHX,-3.0,-59.0,262.0,2153


In [4]:
dates = pd.to_datetime(flight[["year", "month", "day"]])
print(f"Rows: {flight.shape[0]:,} | Columns: {flight.shape[1]}")
print(f"Date range: {dates.min():%Y-%m-%d} to {dates.max():%Y-%m-%d}")
print(f"Origins: {flight.origin.nunique()} | Destinations: {flight.dest.nunique()} | Airlines: {flight.carrier.nunique()}")
print(f"Duplicate records excluding id: {flight.drop(columns="id").duplicated().sum():,}")
print()
flight.info()

Rows: 336,776 | Columns: 21
Date range: 2013-01-01 to 2013-12-31
Origins: 3 | Destinations: 105 | Airlines: 16


Duplicate records excluding id: 0



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 336776 entries, 0 to 336775
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              336776 non-null  int64  
 1   year            336776 non-null  int64  
 2   month           336776 non-null  int64  
 3   day             336776 non-null  int64  
 4   dep_time        328521 non-null  float64
 5   sched_dep_time  336776 non-null  int64  
 6   dep_delay       328521 non-null  float64
 7   arr_time        328063 non-null  float64
 8   sched_arr_time  336776 non-null  int64  
 9   arr_delay       327346 non-null  float64
 10  carrier         336776 non-null  object 
 11  flight          336776 non-null  int64  
 12  tailnum         334264 non-null  object 
 13  origin          336776 non-null  object 
 14  dest            336776 non-null  object 
 15  air_time        327346 non-null  float64
 16  distance        336776 non-null  int64  
 17  hour      

In [5]:
# Data types, missing values, and number of distinct values
column_summary = pd.DataFrame({
    "dtype": flight.dtypes.astype(str),
    "non_null": flight.notna().sum(),
    "missing": flight.isna().sum(),
    "missing_pct": flight.isna().mean().mul(100).round(2),
    "unique_values": flight.nunique()
})
display(column_summary)

,dtype,non_null,missing,missing_pct,unique_values
id,int64,336776,0,0.00,336776
year,int64,336776,0,0.00,1
month,int64,336776,0,0.00,12
day,int64,336776,0,0.00,31
dep_time,float64,328521,8255,2.45,1318
sched_dep_time,int64,336776,0,0.00,1021
dep_delay,float64,328521,8255,2.45,527
arr_time,float64,328063,8713,2.59,1411
sched_arr_time,int64,336776,0,0.00,1163
arr_delay,float64,327346,9430,2.80,577


In [ ]:
# Summary statistics (missing values excluded)
display(flight[["dep_delay", "arr_delay", "air_time", "distance"]].describe().round(2))

,dep_delay,arr_delay,air_time,distance
count,328521.00,327346.00,327346.00,336776.00
mean,12.64,6.90,150.69,1039.91
std,40.21,44.63,93.69,733.23
min,-43.00,-86.00,20.00,17.00
25%,-5.00,-17.00,82.00,502.00
50%,-2.00,-5.00,129.00,872.00
75%,11.00,14.00,192.00,1389.00
max,1301.00,1272.00,695.00,4983.00


### Initial observations
- `origin` and `dest` are categorical location features suitable for the encoding comparison.
- `time_hour` loads as text and should be parsed as a datetime when needed.
- `id` is an identifier, rather than a meaningful numeric predictor.
- Delay and air-time fields contain missing values; account for these before modeling.
- The delay means exceed the medians, indicating that large delays pull the averages upward.